In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os
from google.colab import drive

print("TensorFlow version:", tf.__version__)

# -------------------------------------
# 1️⃣ Mount Google Drive
# -------------------------------------
drive.mount('/content/drive')

# -------------------------------------
# 2️⃣ Load CIFAR-10 dataset
# -------------------------------------
(x_train, y_train), (x_val, y_val) = tf.keras.datasets.cifar10.load_data()

# Normalize images
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0

# Resize images to 224x224 (MobileNet requirement)
resize_layer = tf.keras.Sequential([
    layers.Resizing(224, 224)
])

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .batch(32)
    .map(lambda x, y: (resize_layer(x), y))
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(32)
    .map(lambda x, y: (resize_layer(x), y))
    .prefetch(tf.data.AUTOTUNE)
)

base_mobilenet = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_mobilenet.trainable = False  # freeze base

mobilenet_model = models.Sequential([
    base_mobilenet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")  # CIFAR-10 has 10 classes
])

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n Training MobileNetV2 (frozen base)...")
history_1 = mobilenet_model.fit(train_ds, validation_data=val_ds, epochs=3)

# -------------------------------------
# 4️⃣ Fine-tuning (unfreeze last few layers)
# -------------------------------------
base_mobilenet.trainable = True
for layer in base_mobilenet.layers[:-20]:  # fine-tune last 20 layers
    layer.trainable = False

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # smaller LR
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n Fine-tuning MobileNetV2 (last layers)...")
history_2 = mobilenet_model.fit(train_ds, validation_data=val_ds, epochs=2)

# -------------------------------------
# 5️⃣ Save trained model to Google Drive (.keras)
# -------------------------------------
save_dir = "/content/drive/MyDrive/MobileNet_Model"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "mobilenetv2_cifar10.keras")
mobilenet_model.save(model_path)

print(f"\n Model saved successfully at:\n{model_path}")

# -------------------------------------
# 6️⃣ Verify model load
# -------------------------------------
loaded_model = tf.keras.models.load_model(model_path)
loss, acc = loaded_model.evaluate(val_ds)
print(f"\n Loaded model accuracy: {acc:.4f}")


TensorFlow version: 2.19.0
Mounted at /content/drive
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 207s 1us/step
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

 Training MobileNetV2 (frozen base)...
Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 98s 50ms/step - accuracy: 0.5713 - loss: 1.2524 - val_accuracy: 0.7783 - val_loss: 0.6459
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 107s 37ms/step - accuracy: 0.7652 - loss: 0.6801 - val_accuracy: 0.7969 - val_loss: 0.5831
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 57s 37ms/step - accuracy: 0.7911 - loss: 0.6069 - val_accuracy: 0.8063 - val_loss: 0.5569

 Fine-tuning MobileNetV2 (last layers)...
Epoch 1/2
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 101s 54ms/step - accuracy: 0.7332 - loss: 0.7817 - val_accuracy: 0.8225 - val_loss: 0.5198
Epoch 2/2
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 69s 44ms/step - accuracy: 0.8113 - loss: 0.5476 - val_accuracy: 0.8360 - val_loss: 0.4786

 Model saved successfully at:
/content/drive/MyDrive/MobileNet_Model/mobilenetv2_cifar10.keras
313/313